# Naive Bayes Titanic

Clasificacion Naive Bayes sobre Titanic.

Conversion conceptual 1:1 desde el ejemplo R homologo.


In [ ]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()


def find_data(filename: str) -> Path:
    candidates = [NOTEBOOK_DIR / filename, NOTEBOOK_DIR / "data" / filename]
    for parent in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
        candidates.append(parent / filename)
        candidates.append(parent / "data" / filename)
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"No se encontro el archivo de datos: {filename}")

print("Notebook dir:", NOTEBOOK_DIR)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report

data_path = find_data("titanic.csv")
df = pd.read_csv(data_path)
y = df["Survived"]
X = df[["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]]

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), ["Age", "Fare", "SibSp", "Parch", "Pclass"]),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), ["Sex", "Embarked"]),
], sparse_threshold=0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train_t = pre.fit_transform(X_train)
X_test_t = pre.transform(X_test)
clf = GaussianNB()
clf.fit(X_train_t, y_train)
print(classification_report(y_test, clf.predict(X_test_t)))
